# Proyecto Oráculo · Notebook de búsqueda · Grupo G01

**Análisis y Diseño de Algoritmos · Universidad de Caldas · 2026-2**
Integrante: Jackeline Rivera Aguilera · Semana de entrega: 4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/J2517/ADA-2026-2-GP01/blob/main/busqueda_G01.ipynb)

Este notebook documenta cómo se obtuvo la configuración de `entrega.json`: método, parámetros,
todas las configuraciones evaluadas con su precisión, uso de las trazas de fallos para podar
opciones y validación final con `oraculo.validar`.

| Fase | Qué hace | Evaluaciones (100 instancias c/u) |
|---|---|---|
| 0 · Base | Prompt sin instrucciones | 1 |
| 1 · Cribado guiado por trazas | Cada opción sola; se miden las instancias que rompe y qué familia falló → poda | 5 × 8 = 40 |
| 2 · Ascenso por coordenadas | Sobre las 3 opciones que sobreviven por ranura | ≤ 10 por pasada |
| 3 · Verificación del vecindario | Una pasada con las 8 opciones → certifica óptimo local | ≤ 35 |
| 4 · Temperatura | 0.3 y 0.7 con 3 semillas cada una | 6 |

Total esperado: ≈ 80–100 evaluaciones, frente a las 32 768 de la fuerza bruta.
El resultado es la mejor configuración encontrada y un óptimo local (ningún cambio de una sola
ranura la mejora); no es una garantía de óptimo global (el problema es NP-difícil, ver informe).

**Cómo correrlo:** Entorno de ejecución ▸ Cambiar tipo ▸ T4 GPU → celda 1 → reiniciar sesión → resto en orden.
El caché vive en Google Drive: si Colab se desconecta, volver a correr las celdas reutiliza lo ya generado.

## 1 · Instalar y bajar los archivos del repositorio  ·  *al terminar, reinicia la sesión*

In [ ]:
!pip install -q -U "bitsandbytes>=0.46.1" transformers accelerate \
                   nltk spacy emoji langdetect immutabledict
!python -m spacy download en_core_web_sm -q
!git clone -q https://github.com/allenai/open-instruct

# Versión fija del grupo (fork). --no-cache para no reusar archivos viejos.
REPO = "https://raw.githubusercontent.com/J2517/ADA-2026-2-GP01/main"
for f in ["oraculo.py", "ayudas.py", "datos_visibles.json", "registro.py", "busqueda.py"]:
    !wget -q --no-cache -O {f} {REPO}/{f}

# Datos de nltk a mano (el downloader rechaza el proxy de Colab).
import io, urllib.request, zipfile
NLTK_DATA = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages"
for carpeta, nombre in [("tokenizers", "punkt"), ("tokenizers", "punkt_tab"),
                        ("taggers", "averaged_perceptron_tagger"),
                        ("taggers", "averaged_perceptron_tagger_eng")]:
    with urllib.request.urlopen(f"{NLTK_DATA}/{carpeta}/{nombre}.zip") as resp:
        zipfile.ZipFile(io.BytesIO(resp.read())).extractall(f"/root/nltk_data/{carpeta}")

print("LISTO  →  Entorno de ejecución ▸ Reiniciar sesión, y sigue en la celda 2")

## 2 · El modelo
Qwen3-1.7B en fp16. Requiere GPU T4.

In [ ]:
from ayudas import cargar_modelo
modelo = cargar_modelo("qwen17b")

## 3 · El oráculo, los datos y el registro

- Se busca solo sobre `busqueda` (100 instancias), siempre las mismas y con `semilla=1`,
  para que todas las configuraciones se comparen en igualdad de condiciones (pasos de 1 %).
- `Registro` (en `registro.py`) envuelve al oráculo: anota cada configuración evaluada con su
  precisión, consultas, generaciones nuevas, tiempo y familias violadas.
- `consultas` = costo lógico (instancias consultadas, el presupuesto B del informe).
  `generaciones` = las que de verdad corrieron en GPU en esta sesión (las demás salieron del caché).

In [ ]:
import os
USAR_DRIVE = True   # recomendado: el caché sobrevive a desconexiones

if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA = "/content/drive/MyDrive/oraculo_G01"
else:
    CARPETA = "/content/resultados_G01"
os.makedirs(CARPETA, exist_ok=True)

from oraculo import Oraculo, CATALOGO, espacio
from ayudas import cargar_datos, dividir
from registro import Registro, RANURAS, config_completa
from busqueda import (cribado, podar, inicio_desde_cribado, ascenso,
                      todas_las_opciones, probar_temperaturas)

datos = cargar_datos()
busqueda, validacion = dividir(datos)
oraculo = Oraculo(modelo, busqueda, validacion, cache=f"{CARPETA}/cache_oraculo.json",
                  lote=16, presupuesto=32_000_000)

CONFIGS = espacio()
INSTANCIAS = busqueda
K = len(CATALOGO["rol"])
print(len(CONFIGS), "configuraciones posibles ·", len(INSTANCIAS), "instancias de búsqueda · k =", K)

reg = Registro(oraculo, INSTANCIAS, semilla=1)

## 4 · Búsqueda

### Fase 0 · Línea base y estimación de tiempo
La base es el prompt sin ninguna instrucción (el oráculo permite configuraciones parciales:
una ranura ausente no pega texto). Sirve de referencia para medir qué rompe y qué arregla cada opción.

In [ ]:
base_cfg = {"temperatura": 0.0}
r_base = reg.evaluar(base_cfg, "0-base", "sin instrucciones")
seg = reg.filas[-1]["segundos"]
if reg.filas[-1]["generaciones"]:
    print(f"\nUna evaluación de {len(INSTANCIAS)} instancias tardó {seg/60:.1f} min.")
    print(f"Estimado del cribado (40 evaluaciones): ~{40*seg/3600:.1f} h  "
          "(las opciones que alargan la respuesta tardan más).")

### Fase 1 · Cribado por ranura guiado por trazas
Cada una de las 40 opciones se evalúa sola (las otras cuatro ranuras vacías). Contra la base:

- **roturas**: instancias que la base acertaba y que con esta opción fallan, con la familia de
  restricción que falló según las trazas (`violo`);
- **arreglos**: instancias que la base fallaba y que con esta opción aciertan.

Idea: cada ranura afecta una parte distinta de la respuesta (inicio, largo, estructura, caracteres,
final). Si una opción rompe sistemáticamente familias de esa parte (p. ej. `first_word` para una
opción de `rol` que obliga a empezar con una frase fija), es dañina y se poda.

In [ ]:
r_base, tabla = cribado(reg, K)

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 90)

df_crib = pd.DataFrame([{k: v for k, v in f.items() if k not in ("familias_rotas", "ids_rotas")}
                        for f in tabla])
print(f"Base sin instrucciones: {r_base.precision:.1%}\n")
print("Precisión de cada opción sola (filas = ranura, columnas = índice de opción):")
display(df_crib.pivot(index="ranura", columns="opcion", values="precision")
        .loc[RANURAS].style.format("{:.0%}").background_gradient(axis=1, cmap="RdYlGn"))
print("Roturas (instancias que la base acertaba y esta opción tumba):")
display(df_crib.pivot(index="ranura", columns="opcion", values="roturas").loc[RANURAS])

#### Lectura de trazas por ranura
Para cada opción: su precisión, cuántas instancias rompe, qué familias rompe y un fragmento de
una salida rota (el texto real que generó el modelo). Esta es la evidencia de la poda.

In [ ]:
def ver_roturas(ranura, ejemplos=1, largo=220):
    print(f"══════════ {ranura.upper()} ══════════")
    filas = sorted([f for f in tabla if f["ranura"] == ranura],
                   key=lambda f: (-f["precision"], f["roturas"]))
    for f in filas:
        fams = ", ".join(f"{a} ×{b}" for a, b in f["familias_rotas"].most_common(3)) or "—"
        print(f"\n[{ranura} {f['opcion']}] {f['precision']:.0%} (Δ {f['delta']:+.0%}) · "
              f"roturas {f['roturas']} · arreglos {f['arreglos']}")
        print(f"   texto : {CATALOGO[ranura][f['opcion']][:110]}")
        print(f"   rompe : {fams}")
        r = reg.resultado({ranura: f["opcion"], "temperatura": 0.0})
        for iid in f["ids_rotas"][:ejemplos]:
            salida = next(t["salida"] for t in r.trazas if t["id"] == iid)
            print(f"   ej. id {iid} ({r.fallidas[iid]}): {salida[:largo]!r}")

for ranura in RANURAS:
    ver_roturas(ranura)
    print()

#### Poda
**Criterio:** en cada ranura se ordenan las opciones por precisión (y, en empate, por menos roturas) y se
conservan las 3 mejores; las otras 5 se descartan, dejando escrito qué familias rompían.
Para no confiar ciegamente en esta poda, la Fase 3 vuelve a probar las 8 opciones en contexto.

In [ ]:
CONSERVAR = 3
candidatos, descartes = podar(tabla, conservar=CONSERVAR)
print("Sobreviven:", candidatos)
print(f"Espacio tras la poda: {CONSERVAR}^5 = {CONSERVAR**5} configs (antes {K**5})\n")
display(pd.DataFrame(descartes))

> **Interpretación de las trazas (completar después de correr):**
> - ¿Qué familias rompe cada opción descartada y por qué tiene sentido según su texto?
> - ¿Qué opción de `cierre` arregla más instancias y por qué?
> - ¿Alguna opción sorprendió (rompe familias de otra superficie)?

### Fase 2 · Ascenso por coordenadas sobre las opciones que sobrevivieron
- **Inicio:** la mejor opción de cada ranura en el cribado.
- **Paso:** para cada ranura, en orden `rol → estrategia → formato → estilo → cierre`, se prueban sus
  candidatos con las otras 4 fijas; se cambia solo si la precisión mejora estrictamente.
- **Parada:** una pasada completa sin mejoras (convergencia) o 3 pasadas.

In [ ]:
inicio = inicio_desde_cribado(tabla)
print("Inicio:", inicio)
cfg2, p2, pasadas2, conv2 = ascenso(reg, inicio, candidatos, fase="2-ascenso", max_pasadas=3)
print(f"\nFase 2 → {cfg2}  precisión {p2:.1%}  ({pasadas2} pasadas, convergió={conv2})")

### Fase 3 · Verificación del vecindario completo
El mismo ascenso, pero ahora con las 8 opciones de cada ranura, partiendo de la config de la Fase 2.
Si una pasada completa no mejora nada, la configuración es un óptimo local 1-opt: ninguna
configuración que difiera en una sola ranura la supera. Si algo mejora, la poda de la Fase 1 se
equivocó en contexto (interacción entre ranuras) y el ascenso sigue desde ahí.

In [ ]:
cfg3, p3, pasadas3, conv3 = ascenso(reg, cfg2, todas_las_opciones(K),
                                    fase="3-verificacion", max_pasadas=3)
print(f"\nFase 3 → {cfg3}  precisión {p3:.1%}  ({pasadas3} pasadas, convergió={conv3})")
print("La poda de la Fase 1 se sostuvo." if cfg3 == cfg2 else
      "La verificación encontró una mejora que la poda había descartado.")

### Fase 4 · Temperatura
Con τ > 0 la generación es aleatoria, así que cada τ se mide con 3 semillas. Se cambia a τ > 0 solo si
su precisión promedio supera a la de τ = 0; en empate gana 0.0 (determinista y reproducible).

In [ ]:
cfg4, resumen_t = probar_temperaturas(reg, cfg3, temperaturas=(0.3, 0.7), semillas=(1, 2, 3))
for t, d in resumen_t.items():
    print(f"τ = {t}: media {d['media']:.1%}  desv {d['desv']:.1%}  {['%.0f%%' % (100*p) for p in d['precisiones']]}")

mejor = (resumen_t[cfg4["temperatura"]]["media"], cfg4)
print("\nCONFIGURACIÓN ELEGIDA:", mejor[1], f"· precisión en búsqueda {mejor[0]:.1%}")

## Registro completo de evaluaciones y curva

In [ ]:
import matplotlib.pyplot as plt
from ayudas import curva

df_reg = pd.DataFrame([{**{k: v for k, v in f.items() if k not in ("config", "familias")},
                        **{r: f["config"].get(r, "–") for r in RANURAS},
                        "temperatura": f["config"].get("temperatura"),
                        "familias_mas_violadas": "; ".join(f"{a} ×{b}" for a, b in f["familias"].most_common(3))}
                       for f in reg.filas])
display(df_reg[["n", "fase", *RANURAS, "temperatura", "semilla", "precision", "consultas",
                "generaciones", "segundos", "familias_mas_violadas"]])
print(f"Evaluaciones: {len(reg.filas)} · consultas: {reg.consultas} · "
      f"generaciones nuevas en esta sesión: {reg.generaciones}")
print(f"Fuerza bruta habría necesitado {len(CONFIGS)} × {len(INSTANCIAS)} = {len(CONFIGS)*len(INSTANCIAS):,} consultas")

curva(reg.historial)   # mejor precisión hasta cada consulta (función del curso)

# Mejor precisión contra consultas acumuladas, coloreado por fase
acum, m, xs, ys = 0, 0.0, [], []
for f in reg.filas:
    acum += f["consultas"]; m = max(m, f["precision"]); xs.append(acum); ys.append(m)
plt.figure(figsize=(8, 4))
plt.step(xs, ys, where="post")
for fase in df_reg["fase"].unique():
    sub = df_reg[df_reg["fase"] == fase]
    plt.scatter([xs[i - 1] for i in sub["n"]], sub["precision"], s=12, label=fase)
plt.xlabel("consultas acumuladas"); plt.ylabel("precisión"); plt.legend(); plt.grid(alpha=.3)
plt.title("Precisión de cada evaluación y mejor hasta el momento")
plt.show()

df_reg.to_csv(f"{CARPETA}/registro_busqueda.csv", index=False)
df_reg.to_csv("registro_busqueda.csv", index=False)
df_crib.to_csv("cribado.csv", index=False)
print("Guardado: registro_busqueda.csv y cribado.csv")

## Leer los fallos de la configuración elegida

In [ ]:
from collections import Counter
from ayudas import ver_prompt

r = oraculo.evaluar(mejor[1], INSTANCIAS, semilla=1)   # sale del caché
print(r, "\nFamilias que más fallan:")
for fam, v in Counter(t["violo"] for t in r.trazas).most_common(8):
    print(f"  {v:3}  {fam}")
print()
for t in r.trazas[:3]:
    print("violó:", t["violo"]); print(t["salida"][:300]); print("-" * 60)

print("\nPrompt completo de la primera instancia con la config elegida:\n")
print(ver_prompt(mejor[1], INSTANCIAS[0]))

## Validación final con `oraculo.validar`
Las 87 instancias de validación no se usaron al buscar. Mide si la configuración aguanta instancias
nuevas de las mismas familias; no es la nota: la profesora evalúa sobre un test privado con familias
que no están en este archivo. No se usa para elegir otra configuración.

In [ ]:
r_val = oraculo.validar(mejor[1], n=None)    # n=None → las 87
print("\nbúsqueda (elegida):", f"{mejor[0]:.1%}")
print("validación:        ", f"{r_val.precision:.1%}")

## La entrega

In [ ]:
from ayudas import entrega
from google.colab import files
import shutil

entrega(grupo="G01", config=config_completa(mejor[1]), semana=4)
shutil.copy("entrega.json", f"{CARPETA}/entrega.json")
for f in ["entrega.json", "registro_busqueda.csv", "cribado.csv"]:
    files.download(f)

## (Opcional) Comparación: búsqueda aleatoria con el mismo número de evaluaciones
Sirve para mostrar que el método supera al azar con el mismo gasto. Cuesta tanto como la búsqueda
entera, así que está apagada por defecto. Usa un registro aparte para no mezclar las tablas.

In [ ]:
CORRER_ALEATORIA = False
if CORRER_ALEATORIA:
    from busqueda import busqueda_aleatoria
    reg_azar = Registro(oraculo, INSTANCIAS, semilla=1)
    p_azar, c_azar = busqueda_aleatoria(reg_azar, CONFIGS, evaluaciones=len(reg.filas), semilla_rng=0)
    print(f"Aleatoria ({len(reg_azar.filas)} evals): {p_azar:.1%} {c_azar}")
    print(f"Método G01 ({len(reg.filas)} evals): {mejor[0]:.1%} {mejor[1]}")
    pd.DataFrame([{**f["config"], "precision": f["precision"]} for f in reg_azar.filas]) \
      .to_csv("registro_aleatoria.csv", index=False)